o objetivo desse notebook eh procurar por problemas nos dados


In [0]:
import pandas as pd

# Cole aqui os caminhos copiados
CAMINHO_HIST = "/Workspace/Users/tuanny.leite@gmail.com/Hackathon_Terra_Signal/history.csv"
CAMINHO_INF  = "/Workspace/Users/tuanny.leite@gmail.com/Hackathon_Terra_Signal/inference.csv"

# Lê tudo como texto para não "consertar" nada sem querer (a limpeza vem depois)
hist = pd.read_csv(CAMINHO_HIST, dtype=str, keep_default_na=False)
inf  = pd.read_csv(CAMINHO_INF,  dtype=str, keep_default_na=False)

# Conferência: tem que bater com o arquivo original
print(hist.shape, inf.shape)   # esperado: (5634, 23) (1409, 22)
assert hist.shape == (5634, 23) and inf.shape == (1409, 22), "Leitura ainda quebrada!"
assert hist.customerID.duplicated().sum() == 0

# Sobrescreve as tabelas quebradas com a versão correta
(spark.createDataFrame(hist).write.mode("overwrite")
     .option("overwriteSchema", "true").saveAsTable("workspace.default.history"))
(spark.createDataFrame(inf).write.mode("overwrite")
     .option("overwriteSchema", "true").saveAsTable("workspace.default.inference"))

print("Tabelas regravadas")

In [0]:
hist = spark.table("workspace.default.history").toPandas()
inf  = spark.table("workspace.default.inference").toPandas()

for df in (hist, inf):
    for c in ["SeniorCitizen", "MonthlyCharges", "MonthlyIncome"]:
        df[c] = pd.to_numeric(df[c])
    # tenure e TotalCharges continuam texto de propósito: "unknown" e "null" ainda estão lá para o diagnóstico

In [0]:
def checar_duplicados(df, nome):
    print(f"===== {nome} =====")
    print("Linhas 100% duplicadas:        ", df.duplicated().sum())
    print("customerID duplicado:          ", df["customerID"].duplicated().sum())
    print("Duplicadas ignorando o ID:     ", df.drop(columns="customerID").duplicated().sum())
    print("Feedback de texto duplicado:   ", df["CustomerFeedback"].duplicated().sum())

checar_duplicados(hist, "HISTORY")
checar_duplicados(inf, "INFERENCE")

# Mesmo cliente nas duas bases? (seria vazamento treino/teste)
print("\nIDs em comum history x inference:", len(set(hist.customerID) & set(inf.customerID)))

In [0]:
VALORES_SUSPEITOS = ["", " ", "unknown", "Unknown", "NA", "N/A", "null", "None", "nan", "?"]

def relatorio_nulos(df, nome):
    linhas = []
    for c in df.columns:
        s = df[c]
        nulos = s.isna().sum()
        suspeitos = s.astype(str).str.strip().isin(VALORES_SUSPEITOS).sum() if s.dtype == "object" else 0
        linhas.append({"coluna": c, "tipo": str(s.dtype), "nulos": nulos,
                       "valores_suspeitos": suspeitos,
                       "pct_total": round(100 * (nulos + suspeitos) / len(df), 2)})
    out = pd.DataFrame(linhas).query("nulos > 0 or valores_suspeitos > 0")
    print(f"===== {nome} =====")
    display(out)
    return out

relatorio_nulos(hist, "HISTORY")
relatorio_nulos(inf, "INFERENCE")

In [0]:
VALORES_SUSPEITOS = ["", " ", "unknown", "Unknown", "NA", "N/A", "null", "None", "nan", "?"]

def relatorio_nulos(df, nome):
    linhas = []
    for c in df.columns:
        s = df[c]
        nulos = s.isna().sum()
        suspeitos = s.astype(str).str.strip().isin(VALORES_SUSPEITOS).sum() if s.dtype == "object" else 0
        linhas.append({"coluna": c, "tipo": str(s.dtype), "nulos": nulos,
                       "valores_suspeitos": suspeitos,
                       "pct_total": round(100 * (nulos + suspeitos) / len(df), 2)})
    out = pd.DataFrame(linhas).query("nulos > 0 or valores_suspeitos > 0")
    print(f"===== {nome} =====")
    display(out)
    return out

relatorio_nulos(hist, "HISTORY")
relatorio_nulos(inf, "INFERENCE")

In [0]:
# tenure e TotalCharges vêm como string: ver o que não converte para número
for nome, df in [("HISTORY", hist), ("INFERENCE", inf)]:
    print(f"===== {nome} =====")
    for c in ["tenure", "TotalCharges", "MonthlyCharges", "MonthlyIncome"]:
        conv = pd.to_numeric(df[c].astype(str).str.strip(), errors="coerce")
        nao_num = df.loc[conv.isna(), c].astype(str).value_counts(dropna=False)
        print(f"{c}: {conv.isna().sum()} não numéricos -> {nao_num.to_dict()}")

    t = pd.to_numeric(df["tenure"], errors="coerce")
    print("Linhas com tenure 'unknown' E TotalCharges nulo:",
          (t.isna() & df["TotalCharges"].isna()).sum())
    print("tenure == 0:", (t == 0).sum())

In [0]:
cat_cols = [c for c in hist.select_dtypes(include=["object", "string"]).columns
            if c not in ["customerID", "CustomerFeedback", "tenure", "TotalCharges"]
            and c in inf.columns]

for c in cat_cols:
    vh = hist[c].value_counts(dropna=False).to_dict()
    vi = inf[c].value_counts(dropna=False).to_dict()
    # sinaliza quando há variação só de maiúscula/minúscula/espaço
    normalizado = hist[c].astype(str).str.strip().str.lower().nunique()
    flag = "  <-- INCONSISTENTE" if normalizado < hist[c].nunique() else ""
    print(f"{c}{flag}\n  hist: {vh}\n  inf:  {vi}")

In [0]:
for nome, df in [("HISTORY", hist), ("INFERENCE", inf)]:
    t  = pd.to_numeric(df["tenure"], errors="coerce")
    tc = pd.to_numeric(df["TotalCharges"], errors="coerce")
    ps = df["PhoneService"].str.strip().str.lower()

    print(f"===== {nome} =====")
    # Sem internet -> serviços de internet devem ser 'No internet service'
    print("InternetService=No mas serviço online diferente de 'No internet service':",
          ((df.InternetService == "No") != (df.OnlineSecurity == "No internet service")).sum())
    # Sem telefone -> MultipleLines deve ser 'No phone service'
    print("PhoneService=no x MultipleLines incoerente:",
          ((ps == "no") != (df.MultipleLines == "No phone service")).sum())
    # TotalCharges deveria ser aprox. tenure * MonthlyCharges
    ok = tc.notna() & t.notna() & (t > 0)
    razao = tc[ok] / (t[ok] * pd.to_numeric(df.loc[ok, "MonthlyCharges"], errors="coerce"))
    print("TotalCharges / (tenure*MonthlyCharges):")
    print(razao.describe().round(3).to_string())
    print("Renda mensal menor que a fatura:", (df.MonthlyIncome < df.MonthlyCharges).sum())

In [0]:
palavras = ["churn", "switch", "cancel", "leave", "decided"]
fb = hist["CustomerFeedback"].str.lower()
leak = pd.DataFrame({p: fb.str.contains(p, na=False).groupby(hist["Churn"]).mean().round(2) for p in palavras})
print("% de feedbacks que contêm a palavra, por Churn:")
display(leak)
print("Taxa de churn na base:", round(hist.Churn.eq("Yes").mean(), 3))

In [0]:
import pandas as pd
import numpy as np

def diagnostico(df, nome):
    p = []
    n = len(df)
    t  = pd.to_numeric(df["tenure"], errors="coerce")
    tc = pd.to_numeric(df["TotalCharges"], errors="coerce")
    ps = df["PhoneService"].astype(str)

    def add(problema, coluna, mascara, detalhe=""):
        qtd = int(mascara.sum())
        ex = df.loc[mascara, "customerID"].head(3).tolist()
        p.append({"base": nome, "problema": problema, "coluna": coluna,
                  "linhas": qtd, "pct": round(100 * qtd / n, 2),
                  "exemplos_id": ex, "detalhe": detalhe})

    # 1. Duplicações
    add("Linha duplicada", "todas", df.duplicated())
    add("customerID duplicado", "customerID", df["customerID"].duplicated())
    add("Duplicada ignorando ID", "todas exceto ID", df.drop(columns="customerID").duplicated())

    # 2. Nulos explícitos
    for c in df.columns:
        if df[c].isna().any():
            add("Nulo (NaN)", c, df[c].isna())

    # 3. Nulos disfarçados / tipo errado
    add("Valor 'unknown' em coluna numérica", "tenure",
        df["tenure"].astype(str).str.strip().str.lower() == "unknown",
        "coluna vem como texto por causa disso")
    add("tenure = 0 (cliente novo)", "tenure", t == 0,
        "checar se TotalCharges está nulo nesses casos")
    add("tenure 'unknown' E TotalCharges nulo", "tenure + TotalCharges",
        t.isna() & tc.isna(), "não dá para reconstruir um pelo outro")

    # 4. Categorias inconsistentes
    add("Caixa inconsistente (yes/no minúsculo)", "PhoneService",
        ps.isin(["yes", "no"]), "mesma categoria escrita de 2 jeitos")

    # 5. Regras de negócio
    add("Sem internet mas serviço online preenchido", "InternetService x OnlineSecurity",
        (df.InternetService == "No") != (df.OnlineSecurity == "No internet service"))
    add("Sem telefone x MultipleLines incoerente", "PhoneService x MultipleLines",
        (ps.str.lower() == "no") != (df.MultipleLines == "No phone service"))
    ok = tc.notna() & t.notna() & (t > 0)
    razao = tc / (t * pd.to_numeric(df["MonthlyCharges"], errors="coerce"))
    add("TotalCharges muito fora de tenure*MonthlyCharges (±30%)", "TotalCharges",
        ok & ((razao < 0.7) | (razao > 1.3)), "pode indicar mudança de plano ou erro")
    add("Renda mensal menor que a fatura", "MonthlyIncome", df.MonthlyIncome < df.MonthlyCharges)

    # 6. Texto
    add("Feedback vazio", "CustomerFeedback", df["CustomerFeedback"].fillna("").str.strip() == "")
    add("Feedback duplicado", "CustomerFeedback", df["CustomerFeedback"].duplicated())
    if "Churn" in df.columns:
        add("Feedback cita churn/cancelamento explicitamente", "CustomerFeedback",
            df["CustomerFeedback"].str.lower().str.contains("churn|cancel|switch", na=False),
            "possível leakage: texto pode 'contar' o target")

    return pd.DataFrame(p)

diag = pd.concat([diagnostico(hist, "history"), diagnostico(inf, "inference")])

# Visão lado a lado: só o que tem problema em pelo menos uma base
resumo = (diag.pivot_table(index=["problema", "coluna"], columns="base",
                           values="linhas", aggfunc="sum").fillna(0).astype(int))
resumo = resumo[resumo.sum(axis=1) > 0].sort_values("history", ascending=False)
display(resumo)

# Detalhe com % e exemplos de ID para investigar
display(diag[diag.linhas > 0].sort_values(["base", "linhas"], ascending=[True, False]))

# Distribuição de history vs inference: são comparáveis?
num = ["MonthlyCharges", "MonthlyIncome", "SeniorCitizen"]
display(pd.DataFrame({"history": hist[num].mean(), "inference": inf[num].mean()}).round(2))
print("IDs em comum entre as bases:", len(set(hist.customerID) & set(inf.customerID)))

In [0]:
import pandas as pd, numpy as np

def num(s):
    return pd.to_numeric(s, errors="coerce")

for nome, df in [("HISTORY", hist), ("INFERENCE", inf)]:
    t, tc, mc = num(df.tenure), num(df.TotalCharges), num(df.MonthlyCharges)
    tem_churn = "Churn" in df.columns
    churn = df.Churn.eq("Yes") if tem_churn else None
    print(f"\n==================== {nome} ====================")

    # 1. TotalCharges = "null"
    m = df.TotalCharges.str.strip().str.lower().isin(["null", "", "none", "nan"])
    print(f"\n[1] TotalCharges nulo: {m.sum()} ({m.mean():.1%})")
    print("    tenure desses clientes:", t[m].describe()[["min", "50%", "max"]].to_dict())
    if tem_churn:
        print(f"    churn com nulo: {churn[m].mean():.1%} | sem nulo: {churn[~m].mean():.1%}")

    # 2. tenure = "unknown"
    m = df.tenure.str.strip().str.lower().eq("unknown")
    print(f"\n[2] tenure 'unknown': {m.sum()}")
    print("    TotalCharges disponível para reconstruir:", tc[m].notna().sum(), "de", m.sum())
    if tem_churn:
        print(f"    churn com unknown: {churn[m].mean():.1%} (n pequeno, cuidado)")

    # 3. tenure = 0
    m = t.eq(0)
    print(f"\n[3] tenure = 0: {m.sum()} | TotalCharges desses:", df.TotalCharges[m].unique().tolist())

    # 4. PhoneService em minúsculo
    m = df.PhoneService.isin(["yes", "no"])
    print(f"\n[4] PhoneService minúsculo: {m.sum()} ->", df.PhoneService.value_counts().to_dict())
    if tem_churn:
        print(f"    churn minúsculo: {churn[m].mean():.1%} | resto: {churn[~m].mean():.1%}")

    # 5. TotalCharges fora de tenure*MonthlyCharges
    r = tc / (t * mc)
    m = t.gt(0) & ((r < 0.7) | (r > 1.3))
    cols = ["customerID", "tenure", "MonthlyCharges", "TotalCharges"] + (["Churn"] if tem_churn else [])
    print(f"\n[5] TotalCharges ±30% fora do esperado: {m.sum()}")
    display(df.loc[m, cols].assign(razao=r[m].round(2)))

    # 6. Renda menor que a fatura
    m = num(df.MonthlyIncome) < mc
    print(f"\n[6] Renda < fatura: {m.sum()}")
    if m.any():
        display(df.loc[m, ["customerID", "MonthlyIncome", "MonthlyCharges"]])

# 7. Leakage do texto (só history)
fb = hist.CustomerFeedback.str.lower()
churn = hist.Churn.eq("Yes")
amplo    = fb.str.contains("churn|cancel|switch")
explicit = fb.str.contains("decided to churn|decided to switch|decided to cancel|switching to|have churned")
print("\n[7] LEAKAGE NO TEXTO")
for rotulo, m in [("palavras amplas", amplo), ("frases explícitas", explicit)]:
    print(f"    {rotulo}: {m.mean():.1%} da base | churn quando aparece: {churn[m].mean():.1%} "
          f"| churn quando não aparece: {churn[~m].mean():.1%} | % dos churners cobertos: {m[churn].mean():.1%}")
print("    no inference:", f"{inf.CustomerFeedback.str.lower().str.contains('churn|cancel|switch').mean():.1%} com palavras amplas")

In [0]:
def tratar(df_in):
    df = df_in.copy()
    log = {}

    # 4. Padronizar PhoneService
    log["phone_padronizado"] = df.PhoneService.isin(["yes", "no"]).sum()
    df["PhoneService"] = df.PhoneService.str.strip().str.capitalize()

    # Flags antes de imputar (a ausência carrega sinal de churn)
    df["tenure_missing"] = df.tenure.str.strip().str.lower().eq("unknown").astype(int)
    df["tc_missing"] = df.TotalCharges.str.strip().str.lower().isin(["null", "", "none", "nan"]).astype(int)

    df["tenure"] = num(df.tenure)
    df["TotalCharges"] = num(df.TotalCharges)
    for c in ["MonthlyCharges", "MonthlyIncome", "SeniorCitizen"]:
        df[c] = num(df[c])

    # 3. tenure = 0 -> ainda não pagou nada
    m = df.tenure.eq(0) & df.TotalCharges.isna()
    df.loc[m, "TotalCharges"] = 0.0
    log["tc_zero_cliente_novo"] = m.sum()

    # 2. tenure unknown -> reconstruir por TotalCharges / MonthlyCharges
    m = df.tenure.isna() & df.TotalCharges.notna()
    df.loc[m, "tenure"] = (df.TotalCharges[m] / df.MonthlyCharges[m]).round()
    log["tenure_reconstruido"] = m.sum()

    # 8. Caso sem os dois -> mediana do tenure
    m = df.tenure.isna()
    df.loc[m, "tenure"] = hist_tenure_mediana   # mediana calculada só no history (evita leakage)
    log["tenure_mediana"] = m.sum()

    # 1. TotalCharges nulo -> tenure * MonthlyCharges
    m = df.TotalCharges.isna()
    df.loc[m, "TotalCharges"] = df.tenure[m] * df.MonthlyCharges[m]
    log["tc_imputado"] = m.sum()

    # 6. Feature auxiliar
    df["fatura_sobre_renda"] = df.MonthlyCharges / df.MonthlyIncome

    # 7. Leakage: separar intenção declarada (decidir depois se entra no modelo)
    fb = df.CustomerFeedback.str.lower()
    df["declarou_saida"] = fb.str.contains(
        "decided to churn|decided to switch|decided to cancel|switching to|have churned").astype(int)

    print(log, "| nulos restantes:", df[["tenure", "TotalCharges"]].isna().sum().sum())
    return df

hist_tenure_mediana = num(hist.tenure).median()
hist_tratado = tratar(hist)
inf_tratado  = tratar(inf)

In [0]:
# Salva as bases tratadas como tabelas para usar em outro notebook
(spark.createDataFrame(hist_tratado).write.mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable("workspace.default.history_tratado"))

(spark.createDataFrame(inf_tratado).write.mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable("workspace.default.inference_tratado"))

# Confere o resultado
print(spark.table("workspace.default.history_tratado").count(),
      spark.table("workspace.default.inference_tratado").count())
print("Tabelas salvas: workspace.default.history_tratado e workspace.default.inference_tratado")